# 03. 모델과 메시지 시스템

LangChain v1에서 다양한 LLM 모델을 설정하고, 메시지 타입을 활용하여 대화를 구성하는 방법을 학습합니다.

## 학습 목표

- LangChain v1의 모델 초기화 방법(`init_chat_model`, `ChatOpenAI`)을 이해합니다
- `invoke()`, `stream()`, `batch()` 세 가지 호출 패턴을 학습합니다
- `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` 등 메시지 타입을 이해합니다
- 멀티모달 메시지(이미지 입력)를 구성하는 방법을 익힙니다

## 3.1 환경 설정

`.env` 파일에서 API 키를 로드하고, OpenAI를 통해 모델을 초기화합니다.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# OpenAI를 통한 모델 초기화
model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 초기화 완료:", model.model_name)

모델 초기화 완료: gpt-4.1


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 3.2 모델 프로바이더 비교

LangChain v1은 `init_chat_model()`로 다양한 프로바이더의 모델을 통합된 방식으로 초기화할 수 있습니다.

| 프로바이더 | 모델 문자열 형식 | 필요 패키지 | 환경 변수 |
|-----------|----------------|------------|----------|
| OpenAI | `"openai:gpt-5"` | `langchain-openai` | `OPENAI_API_KEY` |
| Anthropic | `"anthropic:claude-sonnet-4-6"` | `langchain-anthropic` | `ANTHROPIC_API_KEY` |
| Google | `"google:gemini-2.0-flash"` | `langchain-google-genai` | `GOOGLE_API_KEY` |
| AWS Bedrock | `"bedrock:anthropic.claude-v3"` | `langchain-aws` | AWS credentials |
| Azure | `"azure:gpt-4o"` | `langchain-openai` | `AZURE_OPENAI_API_KEY` |
| Ollama | `"ollama:llama3"` | `langchain-ollama` | (로컬 실행) |

> **참고:** OpenAI를 사용하는 경우, `ChatOpenAI`에 `base_url`과 `api_key`를 직접 지정하여 OpenAI API 형식의 서비스(vLLM, LMStudio, Ollama 등)에 접근할 수 있습니다.

## 3.3 init_chat_model() 사용법

`init_chat_model()`은 LangChain v1이 제공하는 통합 모델 초기화 함수입니다.  
프로바이더별 패키지가 설치되어 있으면, 문자열 하나로 모델을 생성할 수 있습니다.

OpenAI를 쓸 때는 `ChatOpenAI`를 직접 사용하는 편이 더 간편합니다.

In [3]:
from langchain.chat_models import init_chat_model

# init_chat_model with OpenAI
model_direct = init_chat_model(
    model="openai:gpt-4.1",
    temperature=0,
)

response = model_direct.invoke("안녕하세요! 2+2는 얼마인가요?", config=lf_config)
print("모델 응답:", response.content)

모델 응답: 안녕하세요! 2+2는 4입니다.


## 3.4 invoke(), stream(), batch() 패턴

LangChain v1의 모든 모델은 세 가지 호출 패턴을 지원합니다:

| 메서드 | 설명 | 반환 타입 |
|--------|------|----------|
| `invoke()` | 단일 입력에 대한 단일 응답 | `AIMessage` |
| `stream()` | 토큰 단위로 스트리밍 응답 | `Iterator[AIMessageChunk]` |
| `batch()` | 여러 입력을 동시에 처리 | `List[AIMessage]` |

In [4]:
# invoke: 단일 호출
response = model.invoke("LangChain이 무엇인지 한 문장으로 설명해 주세요.", config=lf_config)
print("invoke 결과:", response.content)

invoke 결과: LangChain은 다양한 언어 모델과 외부 데이터 소스 및 도구들을 쉽게 연결하고 조합하여 강력한 AI 애플리케이션을 개발할 수 있도록 돕는 오픈소스 프레임워크입니다.


In [5]:
# stream: 토큰 단위 스트리밍
print("stream 결과:", end=" ")
for chunk in model.stream("1부터 5까지 세어 주세요.", config=lf_config):
    print(chunk.content, end="", flush=True)
print()

stream 결과: 

네

,

1

부터

...

3

,

4

,

5

In [6]:
# batch: 여러 입력 동시 처리
responses = model.batch([
    "Python이란 무엇인가요?",
    "JavaScript란 무엇인가요?",
    "Rust란 무엇인가요?",
], config=lf_config)
for i, resp in enumerate(responses):
    print(f"응답 {i+1}: {resp.content[:100]}...")

응답 1: Python은 고급 프로그래밍 언어 중 하나로, 1991년 귀도 반 로썸(Guido van Rossum)이 처음 발표하였습니다. 간결하고 읽기 쉬운 문법(syntax)과 다양한 기...
응답 2: JavaScript란?

**JavaScript**는 웹 페이지를 동적으로 만들기 위해 사용하는 프로그래밍 언어입니다. 1995년 브렌던 아이크(Brendan Eich)가 개발했으...
응답 3: Rust란 무엇인가요?
---

**Rust**는 안전성, 속도, 병행성을 중시하는 시스템 프로그래밍 언어입니다. 주로 Mozilla가 개발을 시작했으며, 오픈소스 커뮤니티에 의해...


## 3.5 메시지 타입

LangChain v1의 메시지 시스템은 대화의 각 역할을 명확히 구분합니다:

| 메시지 타입 | 역할 | 설명 |
|------------|------|------|
| `SystemMessage` | 시스템 | 모델의 행동 방식을 지시하는 시스템 프롬프트 |
| `HumanMessage` | 사용자 | 사용자가 입력하는 메시지 |
| `AIMessage` | AI | 모델이 생성한 응답 |
| `ToolMessage` | 도구 | 도구 실행 결과를 모델에 전달 |

메시지 리스트를 구성하여 `model.invoke()`에 전달하면, 대화 맥락을 유지한 응답을 받을 수 있습니다.

In [7]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage(content="당신은 번역 어시스턴트입니다. 한국어로 번역하세요."),
    HumanMessage(content="안녕하세요, 잘 지내시나요?"),
]

response = model.invoke(messages, config=lf_config)
print("번역 결과:", response.content)
print(f"메시지 타입: {type(response).__name__}")

번역 결과: Hello, how are you doing?

번역: 안녕하세요, 잘 지내시나요?
메시지 타입: AIMessage


In [8]:
# 대화 이력 관리 — 여러 메시지 타입을 조합하여 대화 맥락을 구성합니다
conversation = [
    SystemMessage(content="당신은 수학 튜터입니다. 간결하게 답하세요."),
    HumanMessage(content="소수란 무엇인가요?"),
    AIMessage(content="소수는 1보다 큰 자연수로, 1과 자기 자신 이외의 양의 약수를 가지지 않는 수입니다."),
    HumanMessage(content="5가지 예시를 알려주세요."),
]

response = model.invoke(conversation, config=lf_config)
print("응답:", response.content)

응답: 네, 소수의 예시는 다음과 같습니다:  
2, 3, 5, 7, 11


## 3.6 멀티모달 메시지 (이미지 입력)

LangChain v1에서는 `HumanMessage`의 `content`에 텍스트와 이미지를 함께 전달할 수 있습니다.  
이미지는 URL 또는 base64 인코딩으로 전달하며, 비전(Vision)을 지원하는 모델에서만 동작합니다.

```python
content = [
    {"type": "text", "text": "설명 텍스트"},
    {"type": "image_url", "image_url": {"url": "이미지_URL"}},
]
```

In [9]:
# 멀티모달 메시지 구조 예시 (실행은 모델 지원 필요)
from langchain.messages import HumanMessage

multimodal_msg = HumanMessage(
    content=[
        {"type": "text", "text": "이 이미지에서 무엇이 보이나요?"},
        {
            "type": "image_url",
            "image_url": {"url": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"},
        },
    ]
)

# 멀티모달 지원 모델에서 실행
try:
    response = model.invoke([multimodal_msg], config=lf_config)
    print("멀티모달 응답:", response.content)
except Exception as e:
    print(f"멀티모달 미지원 모델: {e}")
    print("  -> 멀티모달은 GPT-4o, Claude 등 비전 모델에서 지원됩니다.")

멀티모달 미지원 모델: Error code: 400 - {'error': {'message': 'Error while downloading https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_image_url'}}
  -> 멀티모달은 GPT-4o, Claude 등 비전 모델에서 지원됩니다.


## 3.7 요약

이 노트북에서 학습한 핵심 내용:

| 항목 | 설명 |
|------|------|
| `init_chat_model()` | 프로바이더 문자열로 모델을 통합 초기화 |
| `ChatOpenAI(base_url=...)` | OpenAI 등 커스텀 엔드포인트 사용 |
| `invoke()` | 단일 입력 → 단일 응답 |
| `stream()` | 토큰 단위 스트리밍 응답 |
| `batch()` | 여러 입력을 동시에 처리 |
| `SystemMessage` | 시스템 지시사항 설정 |
| `HumanMessage` | 사용자 입력 메시지 |
| `AIMessage` | AI 응답 메시지 (대화 이력용) |
| `ToolMessage` | 도구 실행 결과 전달 |
| 멀티모달 메시지 | `content`에 텍스트와 이미지를 함께 전달 |

### 다음 단계
→ **[04_tools_and_structured_output.ipynb](./04_tools_and_structured_output.ipynb)**: 도구 정의와 구조화된 출력을 배웁니다.
